### Prerequisites

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv(override=True) # Forces immediate pickup of updated .env keys

openai_api_key = os.environ.get('OPENAI_API_KEY', '')
gemini_api_key = os.environ.get('GEMINI_API_KEY', '')

### Define source transcript

In [6]:
source_transcript='''The patient was cooperative but noticeably fatigued today, requiring frequent verbal prompts
to remain engaged. We spent the bulk of the session conducting auditory numeric-recall sequences and
short-term semantic association drills. She expressed deep frustration with remembering short lists of
chores around the house, which she claims is causing tension with her family. I advised her to utilize the
digital voice-recorder tool for her daily tasks at least three times a week and requested that her spouse
attend the next session to discuss collaborative strategies.
'''

### Setup system prompt

In [7]:
system_prompt = """\
You are an expert evaluator of clinical documentation.
Your task is to determine whether a structured clinical summary is a faithful representation of a source therapy transcript.
Evaluate only factual faithfulness.

Do not judge:
- writing style
- grammar
- completeness beyond what is expected in the summary
- formatting

Determine whether the summary:
- accurately reflects statements in the transcript
- contains unsupported claims
- omits clinically important information
- contradicts the transcript

Use the transcript as the sole source of truth.

Evaluate according to:

1. Are all statements supported by the transcript?
2. Are there any contradictions?
3. Are any clinically important facts omitted?
4. Produce an overall faithfulness score from 0.0 to 1.0.

Guidelines for score:

1.0 - No factual errors.
0.9 - Minor wording differences but fully faithful.
0.7 - Some clinically relevant omissions.
0.5 - Several important omissions or minor hallucinations.
0.3 - Major inaccuracies.
0.0 - Fundamentally unfaithful.
"""

### Structured text extraction

#### Ollama

In [ ]:
import ollama

from schemas.clinical_summary import ClinicalSummary

ollama_model = os.environ.get('LLAMA_MODEL_FOR_EXTRACTION', '')

response = ollama.chat(
    model=ollama_model,
    messages=[
        {
            'role': 'system',
            'content': (
                "You are a clinical automation assistant. Extract the relevant information "
                "from the transcript into the requested JSON schema. Be concise and factual. "
                "Only extract metrics, tasks, and symptoms experienced directly by the patient. "
                "Do not extract or attribute actions or complaints made by spouses, family "
                "members, or third parties."
            )
        },
        {
            'role': 'user',
            'content': f"Transcript to parse:\n{source_transcript}"
        }
    ],
    format=ClinicalSummary.model_json_schema()
)

raw_content = response['message']['content']
summary = ClinicalSummary.model_validate_json(raw_content)

#### OpenAI

In [ ]:
from openai import OpenAI
from schemas.clinical_summary import ClinicalSummary

client = OpenAI(api_key=openai_api_key)

resp = client.responses.parse(
    model='gpt-4.1-nano',
    input=[
        {
            "role": "system",
            "content": "Extract structured clinical summaries from transcripts."
        },
        {
            "role": "user",
            "content": source_transcript
        }
    ],
    text_format=ClinicalSummary
)

summary = ClinicalSummary.model_validate_json(resp.output_text)

#### Gemini

In [ ]:
# from google import genai
# from google.genai import types
# from schemas.clinical_summary import ClinicalSummary

# client = genai.Client(api_key=gemini_api_key)

# response = client.models.generate_content(
#     model="gemini-2.5-flash",
#     contents=f"""
# Extract a clinical summary from the following therapy transcript.

# Transcript: {source_transcript}

# """,
#     config=types.GenerateContentConfig(
#         response_mime_type="application/json",
#         response_schema=ClinicalSummary,
#         temperature=0,
#     ),
# )

# summary = ClinicalSummary.model_validate(response.parsed)

#### Print Summary

In [5]:
print(summary.model_dump_json(indent=2))

{
  "patient_mood": "noticably fatigued",
  "exercises_completed": [
    "auditory numeric-recall sequences",
    "short-term semantic association drills"
  ],
  "symptoms_mentioned": [
    "difficulty remembering short lists of chores"
  ],
  "next_steps": "utilize digital voice-recorder tool for daily tasks"
}


### If you want to modify the ClinicalSummary here

In [ ]:
# from schemas.clinical_summary import ClinicalSummary

# summary = ClinicalSummary(
#     patient_mood='cooperative but fatigued', 
#     exercises_completed=[
#         'auditory numeric-recall sequences',
#         'short-term semantic association drills'
#     ], 
#     symptoms_mentioned=['issues with memory'], 
#     next_steps='utilize digital voice-recorder tool 3x a week, collaborate with spouse'
# )

In [ ]:
from typing import Literal

from pydantic import BaseModel, Field

prompt = f"""\
You are an expert evaluator of clinical documentation.
Your task is to determine whether a structured clinical summary is a faithful representation of a source therapy transcript.
Evaluate only factual faithfulness.

Do not judge:
- writing style
- grammar
- completeness beyond what is expected in the summary
- formatting

Determine whether the summary:
- accurately reflects statements in the transcript
- contains unsupported claims
- omits clinically important information
- contradicts the transcript

Use the transcript as the sole source of truth.

Evaluate according to:

1. Are all statements supported by the transcript?
2. Are there any contradictions?
3. Are any clinically important facts omitted?
4. Produce an overall faithfulness score from 0.0 to 1.0.

Guidelines for score:

1.0 - No factual errors.
0.9 - Minor wording differences but fully faithful.
0.7 - Some clinically relevant omissions.
0.5 - Several important omissions or minor hallucinations.
0.3 - Major inaccuracies.
0.0 - Fundamentally unfaithful.

Transcript:
{source_transcript}

Structured summary (JSON):
{summary.model_dump_json(indent=2)}

Return the result using the required schema.
"""

In [ ]:
# response = client.responses.parse(
#     model="gpt-5-mini",
#     input=prompt,
#     text_format=SummaryEvaluation,
# )
# clinical_summary = response.output_parsed


In [ ]:
# from google import genai
# from google.genai import types

# gemini_client = genai.Client(api_key=gemini_api_key)

# response = gemini_client.models.generate_content(
#     model="gemini-2.5-flash",
#     contents=prompt,
#     config=types.GenerateContentConfig(
#         response_mime_type="application/json",
#         response_schema=SummaryEvaluation,
#         temperature=0,
#     ),
# )

# clinical_summary = response.parsed


In [ ]:
from pprint import pprint
import textwrap
from typing import cast
from schemas.clinical_summary import ClinicalSummary

if clinical_summary is None:
    raise ValueError('null value for clinical_summary')

summary = cast(ClinicalSummary, clinical_summary)

print(f"Reasoning:\n\n{'\n'.join(textwrap.wrap(summary.reasoning))}")
print()
print(clinical_summary.model_dump_json(indent=2))

# Make JSON structural checks

In [ ]:
# import json

# def structural_check(summary_json: str) -> dict:
#     data = json.loads(summary_json)
#     required_keys = {"patient_mood", "exercises_completed", "symptoms_mentioned", "next_steps"}
#     missing = required_keys - data.keys()
#     return {"valid_schema": not missing, "missing_keys": list(missing)}

# res = structural_check(summary_json=summary.model_dump_json())

# print(res)

# DeepEval

### Set up test case (DeepEval)

In [ ]:
# from deepeval.test_case import LLMTestCase

# test_case = LLMTestCase(
#     input='Verify source transcript integrity with summary.',
#     actual_output=summary.model_dump_json(),
#     # retrieval_context=[source_transcript],
#     context=[source_transcript]
# )

### Select OpenAI or Gemini as Evaluation LLM Judge

In [ ]:
# import os
# from deepeval.models import GPTModel, GeminiModel

# # model = GeminiModel(model='gemini-2.5-flash', 
# #                     api_key=os.environ.get("GEMINI_API_KEY", ''))

# model = GPTModel(model='gpt-5-mini', 
#                  api_key=os.environ.get("OPENAI_API_KEY", ''))

### Evaluate summary using selected model with DeepEval (GEval)

In [ ]:
# from deepeval.metrics import GEval
# from deepeval.test_case import SingleTurnParams

# summary_str = summary.model_dump_json()

# clinical_accuracy = GEval(
#     name="ClinicalSummaryAccuracy",
#     criteria=(
#         "Determine whether the summary (actual_output) accurately and completely "
#         "captures the mood, exercises performed, symptoms/complaints mentioned, "
#         "and next steps described in the transcript (input), without inventing "
#         "or omitting clinically relevant information."
#     ),
#     evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT, SingleTurnParams.CONTEXT],
#     threshold=0.7,
#     model=model)

# res = await(clinical_accuracy.a_measure(test_case))

# print(f"score: {clinical_accuracy.score}")

### Explain the evaluation

In [ ]:
# from pprint import pprint
# import textwrap

# # with clinical_accuracy (GEval) you use score and reason
# # pprint(clinical_accuracy.criteria)

# print(textwrap.fill(source_transcript, width=90))

# print(f"\nclinical summary: {summary.model_dump_json(indent=2)}\n")

# print(textwrap.fill(clinical_accuracy.reason or '', width=90))